# Quick Select

Encuentra el k-ésimo menor elemento sin ordenar toda la lista. Usa la partición de quick sort, pero solo continúa por el lado donde está el k-ésimo. Cuesta $O(n)$ de media.

## 1. k-ésimo menor

Devolver el k-ésimo menor elemento (k empieza en 1) con pivote aleatorio.

In [1]:
import random

def quick_select(lista, k):
    pivote = random.choice(lista)
    menores = [x for x in lista if x < pivote]
    iguales = [x for x in lista if x == pivote]
    if k <= len(menores):
        return quick_select(menores, k)
    if k <= len(menores) + len(iguales):
        return pivote
    return quick_select([x for x in lista if x > pivote], k - len(menores) - len(iguales))

for _ in range(300):
    datos = random.choices(range(50), k=random.randint(1, 30))
    k = random.randint(1, len(datos))
    assert quick_select(datos, k) == sorted(datos)[k - 1]
print(quick_select([7, 10, 4, 3, 20, 15], 3))

7


## 2. Mediana

Calcular la mediana de una lista con quick select (media de los dos centrales si la longitud es par).

In [2]:
import statistics

def mediana(lista):
    n = len(lista)
    if n % 2:
        return quick_select(lista, n // 2 + 1)
    return (quick_select(lista, n // 2) + quick_select(lista, n // 2 + 1)) / 2

datos = [7, 10, 4, 3, 20, 15]
print(mediana(datos), statistics.median(datos))

8.5 8.5


## 3. k-ésimo menor con `heapq`

`heapq.nsmallest(k, lista)` mantiene un montículo de tamaño $k$ y cuesta $O(n \log k)$. Calcular el k-ésimo menor con él y con un montículo de máximos hecho a mano (valores negados, porque `heapq` solo tiene montículos de mínimos). Comprobar contra `sorted` y comparar tiempos con quick select.

In [3]:
import heapq
import timeit

def k_esimo_heapq(lista, k):
    return heapq.nsmallest(k, lista)[-1]

def k_esimo_monticulo(lista, k):
    # los k menores vistos, con el mayor de ellos en la cima
    monticulo = [-x for x in lista[:k]]
    heapq.heapify(monticulo)
    for x in lista[k:]:
        if x < -monticulo[0]:
            heapq.heapreplace(monticulo, -x)
    return -monticulo[0]

for _ in range(300):
    datos = random.choices(range(50), k=random.randint(1, 30))
    k = random.randint(1, len(datos))
    esperado = sorted(datos)[k - 1]
    assert k_esimo_heapq(datos, k) == k_esimo_monticulo(datos, k) == quick_select(datos, k) == esperado

random.seed(5)
datos = [random.random() for _ in range(200_000)]
metodos = {"nsmallest": k_esimo_heapq, "montículo": k_esimo_monticulo,
           "quick select": quick_select, "sorted": lambda lista, k: sorted(lista)[k - 1]}
for k in (10, len(datos) // 2):
    print(f"k = {k}")
    for nombre, f in metodos.items():
        print(f"  {nombre:13} {timeit.timeit(lambda: f(datos, k), number=1):.4f} s")

k = 10
  nsmallest     0.0031 s
  montículo     0.0049 s
  quick select  0.0154 s
  sorted        0.0237 s
k = 100000
  nsmallest     0.1203 s
  montículo     0.0366 s
  quick select  0.0320 s
  sorted        0.0240 s


## 4. Quick select in situ

Implementar quick select iterativo con la partición de Lomuto, que reordena la propia lista en lugar de crear sublistas ($O(1)$ memoria extra). Comprobar que, al terminar, el k-ésimo queda en su posición con los menores o iguales a su izquierda.

In [4]:
def particion(lista, izq, der):
    p = random.randint(izq, der)
    lista[p], lista[der] = lista[der], lista[p]
    pivote, i = lista[der], izq
    for j in range(izq, der):
        if lista[j] < pivote:
            lista[i], lista[j] = lista[j], lista[i]
            i += 1
    lista[i], lista[der] = lista[der], lista[i]
    return i

def quick_select_in_situ(lista, k):
    izq, der = 0, len(lista) - 1
    while True:
        p = particion(lista, izq, der)
        if p == k - 1:
            return lista[p]
        if p < k - 1:
            izq = p + 1
        else:
            der = p - 1

for _ in range(300):
    datos = random.choices(range(50), k=random.randint(1, 30))
    k = random.randint(1, len(datos))
    copia = datos[:]
    assert quick_select_in_situ(copia, k) == sorted(datos)[k - 1]
    assert sorted(copia) == sorted(datos)
    assert max(copia[:k]) <= copia[k - 1] <= min(copia[k - 1:])

random.seed(6)
datos = [7, 10, 4, 3, 20, 15]
print(quick_select_in_situ(datos, 3), datos)

7 [4, 3, 7, 10, 15, 20]


## 5. Mediana de un flujo de datos

Mantener la mediana mientras llegan números uno a uno con dos montículos de `heapq`: uno de máximos con la mitad menor y otro de mínimos con la mitad mayor, equilibrados para que el primero tenga como mucho uno más. Cada inserción cuesta $O(\log n)$ y consultar la mediana $O(1)$. Comprobarlo con `statistics.median` en cada paso.

In [5]:
class MedianaFlujo:
    def __init__(self):
        self.bajos, self.altos = [], []

    def agregar(self, x):
        heapq.heappush(self.bajos, -x)
        heapq.heappush(self.altos, -heapq.heappop(self.bajos))
        if len(self.altos) > len(self.bajos):
            heapq.heappush(self.bajos, -heapq.heappop(self.altos))

    def mediana(self):
        if len(self.bajos) > len(self.altos):
            return -self.bajos[0]
        return (-self.bajos[0] + self.altos[0]) / 2

random.seed(8)
flujo, vistos = MedianaFlujo(), []
for x in random.choices(range(100), k=500):
    flujo.agregar(x)
    vistos.append(x)
    assert flujo.mediana() == statistics.median(vistos)

flujo = MedianaFlujo()
for x in [5, 15, 1, 3, 8]:
    flujo.agregar(x)
    print(f"llega {x:2} → mediana {flujo.mediana()}")

llega  5 → mediana 5
llega 15 → mediana 10.0
llega  1 → mediana 5
llega  3 → mediana 4.0
llega  8 → mediana 5
